# Final validation: DiMLex-expanded candidates

**This is the last manual validation round for the discourse pipeline.** After
it, the method freezes as either

* **Final A** - standard discopy only, DiMLex as a coverage diagnostic; or
* **Final B** - DiMLex-expanded candidate generation + discopy contextual
  filtering and sense classification.

## The architecture being tested

    candidate set = discopy's own candidates
                    ∪ DiMLex candidates discopy never enumerated
    then the SAME unchanged ConnectiveSenseClassifier decides, for every
    candidate, whether it is a connective and what sense it carries.

DiMLex broadens candidate coverage only. discopy makes every contextual
decision. No per-form rule, whitelist or blacklist exists anywhere in the
pipeline.

All 1,659 DiMLex-only spans were forced through the classifier; 253 were
accepted. This notebook validates those newly accepted spans, because
**false positives are the main risk of adopting the hybrid**.

## What you are judging

> Would this expression count as a PDTB-style Explicit discourse connective
> **in this context**?

Not "does it convey some discourse relation". A construction can express
concession or causality and still not be a PDTB Explicit connective - that
distinction is the whole reason these forms sit outside discopy's inventory.

For `NoSense` controls the sense questions accept `n/a`.

## Why the composition is what it is

The sample deliberately **under-represents `given` / `given that`**, which
already had a dedicated 30-case round, and spreads across the other forms that
contribute accepted mass. `eventually` gets the largest block: it is 98.2%
accepted by the classifier at high confidence, yet your earlier coverage
inspection judged 0/4 of its occurrences valid. If that contradiction holds up
here it is a systematic per-form failure, which is decision criterion 5.

In [1]:
# ============================================================
# Setup
# ============================================================

from pathlib import Path
import re
import sys

import numpy as np
import pandas as pd


def find_repo_root(start=None, repo_name="masters_thesis_sdg"):
    current = (start or Path.cwd()).resolve()
    while True:
        if current.name == repo_name:
            return current
        if current.parent == current:
            raise FileNotFoundError(
                f"Could not find repo root {repo_name!r} above {Path.cwd()}"
            )
        current = current.parent


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.justification_analysis.validation.forced_span_review import (
    ForcedSpanReviewApp, ForcedSpanStore, MANUAL_COLUMNS,
    progress_summary, render_forced_case,
)
from src.justification_analysis.comparison import forced_span_summary as fs

ARTIFACTS = (
    REPO_ROOT / "analysis" / "cross_model" / "base" / "voting" / "prompt_v4"
    / "justification_analysis" / "discourse_parser"
)
PROBE_DIR = ARTIFACTS / "forced_span_probe"
HYBRID_DIR = ARTIFACTS / "experimental_hybrid"

PREDICTIONS_PATH = PROBE_DIR / "forced_span_predictions_all.csv"
SAMPLE_PATH = HYBRID_DIR / "hybrid_validation_sample.csv"
COMPLETED_PATH = HYBRID_DIR / "hybrid_validation_completed.csv"

SEED = 20260827

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("predictions:", PREDICTIONS_PATH.name)
print("sample     :", SAMPLE_PATH.name)
print("answers    :", COMPLETED_PATH.name)

predictions: forced_span_predictions_all.csv
sample     : hybrid_validation_sample.csv
answers    : hybrid_validation_completed.csv


## 1. What the candidate expansion produced

Context before judging: every DiMLex-only span, and what the classifier did
with it.

In [2]:
probe = fs.add_confidence_band(pd.read_csv(PREDICTIONS_PATH, encoding="utf-8-sig"))

print(f"DiMLex-only spans forced: {len(probe):,}")
print(f"accepted                : {int(probe['accepted'].sum()):,} "
      f"({100 * probe['accepted'].mean():.1f}%)")
print(f"NoSense                 : {int(probe['is_nosense'].sum()):,}")

print("\nBy surface form:")
display(fs.acceptance_by(probe, ["form"]).sort_values("n_forced", ascending=False))

print("\nAccepted spans, form x predicted top level:")
accepted = probe.loc[probe["accepted"]]
display(pd.crosstab(accepted["form"], accepted["predicted_top_level"],
                    margins=True))

print("\nConfidence by outcome:")
display(fs.confidence_summary(probe))
print(f"accepted mean confidence: {accepted['confidence'].mean():.3f}")
print(f"NoSense  mean confidence: "
      f"{probe.loc[~probe['accepted'], 'confidence'].mean():.3f}")

DiMLex-only spans forced: 1,659
accepted                : 253 (15.3%)
NoSense                 : 1,406

By surface form:


,form,n_forced,n_accepted,n_nosense,acceptance_rate_pct,mean_confidence
7,with,834,40,794,4.8,0.911208
2,given,477,50,427,10.5,0.943629
0,despite,113,45,68,39.8,0.669279
5,particularly,74,0,74,0.0,0.967008
1,eventually,57,56,1,98.2,0.975827
3,given that,54,53,1,98.1,0.475540
8,without,36,7,29,19.4,0.853350
6,upon,12,2,10,16.7,0.992999
4,in response to,2,0,2,0.0,0.977271



Accepted spans, form x predicted top level:


predicted_top_level,Comparison,Contingency,Expansion,Temporal,All
form,,,,,
despite,45,0,0,0,45
eventually,0,0,0,56,56
given,0,50,0,0,50
given that,0,41,0,12,53
upon,0,0,0,2,2
with,1,0,4,35,40
without,1,0,1,5,7
All,47,91,5,110,253



Confidence by outcome:


accepted,NoSense,accepted
confidence_band,,
<0.50,54,96
0.50-0.75,71,93
0.75-0.90,81,7
>=0.90,1200,57


accepted mean confidence: 0.620
NoSense  mean confidence: 0.943


## 2. The sample

~22 cases: accepted candidates weighted toward the forms that contribute new
mass, plus `NoSense` controls to check that obvious lexical noise is still
rejected. Fixed seed; the exact rows are written to CSV so the sample is
reproducible independently of this notebook.

In [3]:
# Quotas by (form, prediction). `given` / `given that` are capped at one each
# because they already had a dedicated round; the remaining accepted forms are
# covered as fully as the data allows. NoSense controls span the forms most
# likely to be lexical noise.
ACCEPTED_QUOTAS = {
    "eventually": 5,     # 98.2% accepted, but 0/4 judged valid earlier
    "despite": 4,        # 39.8% accepted, 1/7 judged valid earlier
    "with": 4,           # largest DiMLex-only form, 4.8% accepted
    "without": 2,
    "upon": 2,
    "given": 1,          # already validated separately
    "given that": 1,     # already validated separately
}
NOSENSE_QUOTAS = {
    "particularly": 2,   # 0% accepted - control that noise stays rejected
    "with": 2,
    "despite": 1,
    "given": 1,
}


def build_sample(frame, seed=SEED):
    """Fixed-seed quota sample, spread within each cell."""
    rng = np.random.RandomState(seed)
    picked = []

    def take(cell, quota):
        chosen = []
        for column in ("model", "decoding_group", "predicted_sense",
                       "confidence_band"):
            for value in cell[column].dropna().unique():
                if len(chosen) >= quota:
                    break
                pool = cell.loc[cell[column].eq(value) & ~cell.index.isin(chosen)]
                if len(pool):
                    chosen.append(pool.sample(1, random_state=rng).index[0])
        leftover = cell.loc[~cell.index.isin(chosen)]
        if len(chosen) < quota and len(leftover):
            chosen += list(leftover.sample(min(quota - len(chosen), len(leftover)),
                                           random_state=rng).index)
        return chosen[:quota]

    for form, quota in ACCEPTED_QUOTAS.items():
        cell = frame.loc[frame["form"].eq(form) & frame["accepted"]]
        picked += take(cell, quota)
    for form, quota in NOSENSE_QUOTAS.items():
        cell = frame.loc[frame["form"].eq(form) & ~frame["accepted"]
                         & ~frame.index.isin(picked)]
        picked += take(cell, quota)

    return frame.loc[picked].reset_index(drop=True)


sample = build_sample(probe)
sample["provenance"] = "dimlex_expanded"
for column in MANUAL_COLUMNS:
    sample[column] = ""

SAMPLE_PATH.parent.mkdir(parents=True, exist_ok=True)
sample.to_csv(SAMPLE_PATH, index=False, encoding="utf-8-sig")

print(f"sampled: {len(sample)} cases (seed {SEED})")
print(f"  accepted: {int(sample['accepted'].sum())}   "
      f"NoSense: {int((~sample['accepted']).sum())}")
print(f"saved -> {SAMPLE_PATH}")

print("\nform x prediction:")
display(sample.assign(
    prediction=np.where(sample["accepted"], "accepted", "NoSense")
).groupby(["form", "prediction"], observed=True).size().rename("n").reset_index())

print("\ncoverage:")
display(sample["model"].value_counts().rename_axis("model").reset_index(name="n"))
display(sample["decoding_group"].value_counts()
        .rename_axis("decoding").reset_index(name="n"))
display(sample["predicted_sense"].value_counts()
        .rename_axis("predicted_sense").reset_index(name="n"))
display(sample["confidence_band"].value_counts()
        .rename_axis("confidence_band").reset_index(name="n"))

sampled: 25 cases (seed 20260827)
  accepted: 19   NoSense: 6
saved -> C:\Users\annab\Documents\GitHub\masters_thesis_sdg\analysis\cross_model\base\voting\prompt_v4\justification_analysis\discourse_parser\experimental_hybrid\hybrid_validation_sample.csv

form x prediction:


,form,prediction,n
0,despite,NoSense,1
1,despite,accepted,4
2,eventually,accepted,5
3,given,NoSense,1
4,given,accepted,1
5,given that,accepted,1
6,particularly,NoSense,2
7,upon,accepted,2
8,with,NoSense,2
9,with,accepted,4



coverage:


,model,n
0,Gemma 4 2B,12
1,Gemma 4 4B,7
2,Gemma 4 31B,6


,decoding,n
0,Stochastic,21
1,Greedy,4


,predicted_sense,n
0,Temporal.Asynchronous,8
1,NoSense,6
2,Temporal.Synchrony,5
3,Comparison.Contrast,4
4,Contingency.Cause,2


,confidence_band,n
0,>=0.90,11
1,<0.50,7
2,0.50-0.75,5
3,0.75-0.90,2


## 3. Review

One case at a time. The span is highlighted from its exact offsets; the
classifier's prediction, confidence and provenance are shown.

1. **Is this a valid PDTB-style Explicit connective in this context?**
2. **Is the top-level sense correct?** — `n/a` for `NoSense` controls
3. **Is the full sense reasonable?** — `n/a` for `NoSense` controls
4. **Expected top-level category**, if it is a connective
5. **Expected full sense** and **notes**, optional

Every control saves immediately.

In [4]:
cases = sample.to_dict("records")

for case in cases:
    sentence = str(case.get("sentence_text", ""))
    match = re.search(rf"(?<!\w){re.escape(str(case['marker']))}(?!\w)",
                      sentence, re.I)
    case["sent_start"] = match.start() if match else -1
    case["sent_end"] = match.end() if match else -1

store = ForcedSpanStore(COMPLETED_PATH, cases)

assert len({c["probe_id"] for c in cases}) == len(cases), "duplicate probe_id"
assert 20 <= len(cases) <= 25, f"unexpected sample size {len(cases)}"

print(f"cases loaded    : {len(cases)}")
print(f"already answered: {store.n_answered()} / {len(cases)}")

app = ForcedSpanReviewApp(cases, store)
display(app.ui)

cases loaded    : 25
already answered: 0 / 25


## 4. Progress and results

Results appear once every case is answered. Raw counts only - the sample is
quota-based for coverage, so it supports counts and per-form patterns, not a
rate.

In [5]:
store.save()

answers = store.to_frame()
n_answered = store.n_answered()

print(f"answered: {n_answered} / {len(cases)}")
print(f"saved -> {COMPLETED_PATH}")
display(progress_summary(cases, store))

if n_answered < len(cases):
    print("\nstill to review (probe_id): "
          f"{[int(c['probe_id']) for c in cases if not store.is_answered(c)]}")
else:
    valid = answers["manual_is_explicit_connective"].str.lower().eq("yes")
    acc = answers["accepted"].astype(str).str.lower().isin(["true", "1"])

    print(f"\njudged valid Explicit connectives: {int(valid.sum())}/{len(answers)}")
    print(f"  among ACCEPTED cases : {int((valid & acc).sum())}/{int(acc.sum())}")
    print(f"  among NoSense controls: {int((valid & ~acc).sum())}/{int((~acc).sum())}"
          "   (high = the classifier wrongly rejected them)")

    print("\nBy surface form (accepted cases only):")
    display(
        answers.loc[acc].assign(valid=valid.loc[acc])
        .groupby("form", observed=True)
        .agg(n=("valid", "size"), judged_valid=("valid", "sum"))
        .assign(judged_invalid=lambda t: t["n"] - t["judged_valid"])
    )

    top = answers.loc[acc & answers["manual_top_level_correct"]
                      .str.lower().isin(["yes", "no"])]
    if len(top):
        print(f"\ntop-level sense correct: "
              f"{int(top['manual_top_level_correct'].str.lower().eq('yes').sum())}"
              f"/{len(top)}")
    full = answers.loc[acc & answers["manual_full_sense_correct"]
                       .str.lower().isin(["yes", "no"])]
    if len(full):
        print(f"full sense reasonable  : "
              f"{int(full['manual_full_sense_correct'].str.lower().eq('yes').sum())}"
              f"/{len(full)}")

    print("\nRaw counts only; quota sample, so no rate is implied.")

answered: 0 / 25


saved -> C:\Users\annab\Documents\GitHub\masters_thesis_sdg\analysis\cross_model\base\voting\prompt_v4\justification_analysis\discourse_parser\experimental_hybrid\hybrid_validation_completed.csv


,form,prediction,n,answered,remaining
0,despite,NoSense,1,0,1
1,despite,accepted,4,0,4
2,eventually,accepted,5,0,5
3,given,NoSense,1,0,1
4,given,accepted,1,0,1
5,given that,accepted,1,0,1
6,particularly,NoSense,2,0,2
7,upon,accepted,2,0,2
8,with,NoSense,2,0,2
9,with,accepted,4,0,4



still to review (probe_id): [677, 913, 1350, 1428, 784, 734, 1013, 1525, 1085, 623, 1036, 1250, 557, 779, 1189, 1458, 1554, 11, 34, 652, 1007, 743, 1003, 576, 121]


### Static view (optional)

In [6]:
from IPython.display import HTML, display as _display

CASE_TO_SHOW = 1

_display(HTML(render_forced_case(
    cases[CASE_TO_SHOW - 1], CASE_TO_SHOW, len(cases)
)))

## 5. Decision criterion

The hybrid becomes the final method **only if all five hold**:

1. newly accepted DiMLex-only candidates are overwhelmingly genuine
   PDTB-style Explicit connectives;
2. their top-level senses are overwhelmingly correct;
3. finer-grained senses are reliable enough for the secondary analysis;
4. obvious lexical noise continues to be rejected (the `NoSense` controls);
5. **no single lexical form introduces a systematic error pattern.**

If they hold, adopt the hybrid **even though its numerical impact is modest**
(+253 relations, +4.6%), because the architecture is cleaner: broader candidate
recall with one contextual classifier deciding everything.

If they do not - particularly if condition 5 fails - keep standard discopy and
use DiMLex purely as a coverage diagnostic.

Nothing in the production pipeline changes on the basis of this notebook alone.